# Chapter 9. Introduction to Artificial Neural Networks

Artificial Neural Networks (ANNs) are machine learning models inspired by networks of biological neurons.

They are the foundation of deep learning and are widely used for complex tasks such as image classification, speech recognition, recommendation systems, and language models.

In this chapter, we will first explore early neural network architectures and then build simple multilayer perceptrons using Scikit-Learn.

Neural networks became practical because of several factors:

- large amounts of training data
- increased computing power, especially GPUs
- improved training algorithms
- modern architectures such as Transformers

Although ANNs were inspired by biological neurons, they are mathematical models and do not need to closely imitate the brain.

## The Perceptron

The perceptron is one of the earliest neural network architectures.

It learns connection weights to make predictions from input features.

A perceptron uses threshold logic units (TLUs).

Each TLU computes a weighted sum of the inputs plus a bias, then applies a step function:

$$
h_w(x) = \text{step}(w^\top x + b)
$$

A perceptron can contain multiple TLUs in one fully connected layer.

It can learn linear decision boundaries, so it cannot solve nonlinearly separable problems such as XOR.

For a batch of instances, a fully connected layer can be written as:

$$
Y = \phi(XW + b)
$$

where $W$ contains the weights, $b$ contains the biases, and $\phi$ is the activation function.

During training, the perceptron updates its weights when it makes a wrong prediction.

If the data is linearly separable, the perceptron learning algorithm is guaranteed to find a separating boundary.

In [31]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.linear_model import Perceptron

iris = load_iris(as_frame=True)

X = iris.data[["petal length (cm)", "petal width (cm)"]].values
# Binary target: Iris setosa or not
y = (iris.target == 0)

per_clf = Perceptron(random_state=42)

per_clf.fit(X, y)

X_new = [[2, 0.5], [3, 1]]

y_pred = per_clf.predict(X_new) # predicts True and False for these 2 flowers

print("Predictions:")
print(y_pred)

Predictions:
[ True False]


Unlike logistic regression, a perceptron does not output class probabilities.

Its linear limitation motivates the use of multilayer perceptrons.

## The Multilayer Perceptron and Backpropagation

A multilayer perceptron (MLP) stacks multiple layers of neurons.

Unlike a single-layer perceptron, an MLP can learn nonlinear patterns such as XOR.

An MLP has:

- an input layer
- one or more hidden layers
- an output layer

Since information flows only from the inputs to the outputs, it is a feedforward neural network.

Hidden layers allow the network to combine simple patterns into more complex ones.

For example, a single perceptron cannot solve XOR because XOR is not linearly separable. However, an MLP can combine multiple linear boundaries to solve it.

When a neural network has many hidden layers, it is called a deep neural network (DNN).

Deep networks can learn hierarchical representations, where lower layers learn simple features and higher layers combine them into more complex features.

### Backpropagation

Backpropagation is the main training algorithm for neural networks.

It uses reverse-mode automatic differentiation and gradient descent to reduce the model's error.

For each mini-batch, backpropagation follows these steps:

1. Compute predictions through the network.
2. Measure the prediction error using a loss function.
3. Compute gradients of the loss with respect to all weights and biases.
4. Update the parameters using gradient descent.

The first step is called the forward pass.

The gradient computation moves from the output layer back toward the input layer, so it is called the backward pass.

Training is usually performed using mini-batches.

One complete pass through the training set is called an epoch.

Hidden-layer weights must be initialized randomly.

If all weights were initialized to the same value, neurons in the same layer would learn the same features and the network would behave as if it had only one neuron per layer.

### Activation Functions

Backpropagation requires differentiable activation functions.

The step function used by perceptrons is not suitable because its gradient is zero almost everywhere.

The sigmoid activation function is:

$$
\sigma(z) = \frac{1}{1 + \exp(-z)}
$$

Its output ranges from $0$ to $1$ and it has a nonzero derivative, allowing gradient descent to update the model parameters.

Other common activation functions include:

- `tanh`: outputs values between $-1$ and $1$
- ReLU: outputs $0$ for negative inputs and returns the input for positive values

ReLU is fast to compute and is a common default for hidden layers.

The ReLU activation function is:

$$
\text{ReLU}(z) = \max(0, z)
$$

Activation functions add nonlinearity to neural networks.

Without nonlinear activations, multiple layers of linear transformations would still be equivalent to a single linear transformation.

This is why nonlinear activation functions are necessary for learning complex patterns.

## Building and Training MLPs with Scikit-Learn

Scikit-Learn provides `MLPRegressor` for regression tasks and `MLPClassifier` for classification tasks.

Neural networks are sensitive to feature scales, so input preprocessing is important.

### Regression MLPs

For regression, the number of output neurons depends on the number of target values.

A single-target regression task needs one output neuron.

In [50]:
from sklearn.datasets import fetch_california_housing
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

In [52]:
housing = fetch_california_housing()

X_train, X_test, y_train, y_test = train_test_split(
    housing.data, housing.target, random_state=42)

We create an MLP with three hidden layers, each containing 50 neurons.

The model uses ReLU in the hidden layers and a linear output layer for regression.

In [60]:
mlp_reg = MLPRegressor(hidden_layer_sizes=[50, 50, 50], early_stopping=True,
                       verbose=False, random_state=42)

With `early_stopping=True`, the model sets aside part of the training data for validation.

Training stops when the validation score does not improve for several epochs.

In [67]:
pipeline = make_pipeline(StandardScaler(), mlp_reg)

pipeline.fit(X_train, y_train)

print("Number of iterations:", mlp_reg.n_iter_)
print("\nBest validation score:", mlp_reg.best_validation_score_)
print("\nFinal training loss:", mlp_reg.loss_)

Number of iterations: 45

Best validation score: 0.7915361254257781

Final training loss: 0.12960481184645378


`StandardScaler` is important because gradient-based optimization works better when input features have similar scales.

For regression, `MLPRegressor.score()` returns the $R^2$ score.

A higher $R^2$ value means that the model explains more of the variation in the target values.

In [81]:
y_pred = pipeline.predict(X_test)

rmse = root_mean_squared_error(y_test, y_pred)

print("Test RMSE:", rmse)

Test RMSE: 0.5327699946812925


The model minimizes mean squared error during training.

Since neural networks can overfit, early stopping and regularization are useful for improving generalization.